# Concatination Transformation Text

Validates the mathematical correctness of assembling independent superoperators into a direct-sum (block-diagonal) Liouville basis. Ensures that transformation routines correctly preserve block structure, state vectorization conventions, and numerical accuracy when concatenating subsystems of varying dimensions.

Specific Checks:
1. Correct block-diagonal structure of the combined superoperator
2. Accurate application of the transformed superoperator to block-diagonal density matrices
3. Preservation of row-major vectorization conventions during reshaping
4. Compatibility with single-subsystem edge cases (identity-like behavior)
5. Numerical consistency across heterogeneous subsystem dimensions (2×2, 3×3, 1×1)

In [1]:
import torch
import numpy as np
from typing import List

import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from mars.population.transform import reshape_superoperators_list_to_direct_sum_basis, Liouvilleator

In [2]:
import torch
import math

def test_reshape_superoperators_tensor_to_direct_sum_basis():
    torch.manual_seed(42)

    # ----- Test 1: two 2×2 subsystems -----
    N1, N2 = 2, 2
    R1 = torch.randn(N1 * N1, N1 * N1)
    R2 = torch.randn(N2 * N2, N2 * N2)
    superops = [R1, R2]
    total_R = reshape_superoperators_list_to_direct_sum_basis(superops)

    N = N1 + N2
    assert total_R.shape == (N * N, N * N)

    rho1 = torch.randn(N1, N1)
    rho2 = torch.randn(N2, N2)
    rho_direct = torch.block_diag(rho1, rho2)
    vec_rho = Liouvilleator.vec(rho_direct)

    vec_rho_prime = total_R @ vec_rho

    # Expected result
    vec_rho1 = Liouvilleator.vec(rho1)
    vec_rho2 = Liouvilleator.vec(rho2)
    vec_rho1_prime = R1 @ vec_rho1
    vec_rho2_prime = R2 @ vec_rho2
    rho1_prime = Liouvilleator.unvec(vec_rho1_prime)
    rho2_prime = Liouvilleator.unvec(vec_rho2_prime)
    rho_expected = torch.block_diag(rho1_prime, rho2_prime)
    vec_expected = Liouvilleator.vec(rho_expected)

    assert torch.allclose(vec_rho_prime, vec_expected, atol=1e-6)

    # ----- Test 2: single subsystem -----
    R = torch.randn(3 * 3, 3 * 3)          # N = 3
    total_R = reshape_superoperators_list_to_direct_sum_basis([R])
    N = 3
    assert total_R.shape == (N * N, N * N)

    rho = torch.randn(N, N)
    vec_rho = Liouvilleator.vec(rho)
    vec_rho_prime = total_R @ vec_rho
    expected = R @ vec_rho
    assert torch.allclose(vec_rho_prime, expected)

    # ----- Test 3: three subsystems of different sizes -----
    N1, N2, N3 = 2, 3, 1
    R1 = torch.randn(N1 * N1, N1 * N1)
    R2 = torch.randn(N2 * N2, N2 * N2)
    R3 = torch.randn(N3 * N3, N3 * N3)
    superops = [R1, R2, R3]
    total_R = reshape_superoperators_list_to_direct_sum_basis(superops)

    N = N1 + N2 + N3
    assert total_R.shape == (N * N, N * N)

    # Random block‑diagonal state
    rho1 = torch.randn(N1, N1)
    rho2 = torch.randn(N2, N2)
    rho3 = torch.randn(N3, N3)
    rho_direct = torch.block_diag(rho1, rho2, rho3)
    vec_rho = Liouvilleator.vec(rho_direct)
    vec_rho_prime = total_R @ vec_rho

    # Expected
    vec_rho1 = Liouvilleator.vec(rho1)
    vec_rho2 = Liouvilleator.vec(rho2)
    vec_rho3 = Liouvilleator.vec(rho3)
    vec_rho1_prime = R1 @ vec_rho1
    vec_rho2_prime = R2 @ vec_rho2
    vec_rho3_prime = R3 @ vec_rho3
    rho1_prime = Liouvilleator.unvec(vec_rho1_prime)
    rho2_prime = Liouvilleator.unvec(vec_rho2_prime)
    rho3_prime = Liouvilleator.unvec(vec_rho3_prime)
    rho_expected = torch.block_diag(rho1_prime, rho2_prime, rho3_prime)
    vec_expected = Liouvilleator.vec(rho_expected)

    assert torch.allclose(vec_rho_prime, vec_expected, atol=1e-6)

    print("All tests passed.")

if __name__ == "__main__":
    test_reshape_superoperators_tensor_to_direct_sum_basis()

All tests passed.
